# Model search

Goal:
Test stronger classification models using cross-validation on the training data only.

We will compare:
- Logistic Regression
- Logistic Regression with class_weight="balanced"
- Random Forest
- Extra Trees
- Gradient Boosting
- HistGradientBoosting
- SVC with RBF kernel

The final X_test set is not used during model selection.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: c:\temp\python_learning\ml_projects\diabetes_predictions\notebooks
Project root: c:\temp\python_learning\ml_projects\diabetes_predictions


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC

from src.data_utils import load_data, split_features_target, make_train_test_split
from src.preprocessing import replace_suspicious_zeros_with_indicators

from src.evaluation import make_stratified_cv, get_classification_scoring

cv = make_stratified_cv()
scoring = get_classification_scoring()

print(cv)
print(scoring.keys())

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
dict_keys(['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])


In [5]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes.csv"

df = load_data(DATA_PATH)
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = make_train_test_split(X, y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (614, 8)
X_test shape: (154, 8)

Train target distribution:
Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test target distribution:
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
}

In [7]:
from src.pipelines import build_classification_pipeline

In [11]:
from sklearn.linear_model import LogisticRegression
from src.pipelines import build_classification_pipeline

test_pipeline = build_classification_pipeline(
    model=LogisticRegression(max_iter=1000),
    use_scaler=True,
    use_indicators=True,
)

test_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('zero_handler', ...), ('imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function rep...0020777F314E0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments t

In [12]:
model_configs = {
    "logreg": {
        "model": LogisticRegression(max_iter=1000),
        "use_scaler": True,
    },
    "logreg_balanced": {
        "model": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "use_scaler": True,
    },
    "random_forest": {
        "model": RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        ),
        "use_scaler": False,
    },
    "extra_trees": {
        "model": ExtraTreesClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        ),
        "use_scaler": False,
    },
    "gradient_boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "use_scaler": False,
    },
    "hist_gradient_boosting": {
        "model": HistGradientBoostingClassifier(random_state=42),
        "use_scaler": False,
    },
    "svc_rbf": {
        "model": SVC(
            kernel="rbf",
            probability=True,
            random_state=42,
        ),
        "use_scaler": True,
    },
}

In [13]:
model_search_results = []

for model_name, config in model_configs.items():
    pipeline = build_classification_pipeline(
        model=config["model"],
        use_scaler=config["use_scaler"],
        use_indicators=True,
    )

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
    )

    model_search_results.append(
        {
            "model": model_name,
            "accuracy_mean": cv_results["test_accuracy"].mean(),
            "accuracy_std": cv_results["test_accuracy"].std(),
            "precision_mean": cv_results["test_precision"].mean(),
            "recall_mean": cv_results["test_recall"].mean(),
            "f1_mean": cv_results["test_f1"].mean(),
            "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        }
    )

model_search_results_df = pd.DataFrame(model_search_results)
model_search_results_df.sort_values("f1_mean", ascending=False)

,model,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean
1,logreg_balanced,0.763841,0.005200,0.645921,0.724474,0.680645,0.844114
0,logreg,0.789871,0.014476,0.760518,0.588815,0.661436,0.843462
2,random_forest,0.762202,0.015949,0.694805,0.579513,0.629656,0.817488
4,gradient_boosting,0.752392,0.024998,0.666820,0.598007,0.627151,0.823158
6,svc_rbf,0.768746,0.007815,0.718920,0.560908,0.626348,0.829581
5,hist_gradient_boosting,0.750833,0.030089,0.666061,0.584275,0.620794,0.800386
3,extra_trees,0.755738,0.022622,0.686448,0.561019,0.615593,0.826946


In [14]:
model_search_results_df.sort_values("recall_mean", ascending=False)

,model,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean
1,logreg_balanced,0.763841,0.005200,0.645921,0.724474,0.680645,0.844114
4,gradient_boosting,0.752392,0.024998,0.666820,0.598007,0.627151,0.823158
0,logreg,0.789871,0.014476,0.760518,0.588815,0.661436,0.843462
5,hist_gradient_boosting,0.750833,0.030089,0.666061,0.584275,0.620794,0.800386
2,random_forest,0.762202,0.015949,0.694805,0.579513,0.629656,0.817488
3,extra_trees,0.755738,0.022622,0.686448,0.561019,0.615593,0.826946
6,svc_rbf,0.768746,0.007815,0.718920,0.560908,0.626348,0.829581
